# Results viewer

This notebook loads the precomputed results in `results/` and plots them. It needs only
NumPy and Matplotlib, so it runs without PySEQM or a GPU.

1. **XL-BOMD trajectories** (AM1, PySEQM): RMSD, per-atom RMSF, NVE energy conservation,
   potential energy vs RMSD, and a 3D view of a frame.
2. **QLSA experiment**: QSVT linear solve of the linearized XL-BOMD system, compared with the
   classical step-by-step solution (condition number, query count, deviation, surrogate drift).
3. **Periodic re-linearization**: the surrogate refreshed with the SCF density every W steps.
4. **MAE evaluation**: AM1 vs DFT energy and force errors, next to published ML-potential errors.

### Main findings

* QSVT and classical solutions agree to 1e-13 to 2e-11 (relative Frobenius norm) on AM1 densities.
* With dissipation (Niklasson Eq. 24, α = 0.5, γ = 0.5) the effective condition number stays at
  about 29 for windows up to T = 4000; without dissipation it grows linearly (592 at T = 400).
* A single linearization drifts from the SCF densities by 3-4 % after 25 steps and 11-14 % after 200.
* Re-linearizing every 5-10 steps keeps the deviation at 1-2 % over 250 fs, without accumulation.
* AM1 errors against the DFT labels are 10-100 times larger than those of trained ML potentials.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# repository root, whether the notebook is started from notebooks/ or from the root
ROOT = Path.cwd()
while not (ROOT / "results").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OBS, MD, QLSA, MAE = (ROOT / "results" / d for d in ("observables", "md", "qlsa", "mae"))

EV_TO_KCAL = 23.060548867
SYSTEMS = {"iso17": "ISO17 (C$_7$O$_2$H$_{10}$)", "md17": "MD17 aspirin (C$_9$H$_8$O$_4$)"}
ELEMENT_COLORS = {"C": "#2C3E50", "O": "#C0392B", "H": "#27AE60", "N": "#2471A3"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2,
                     "axes.spines.top": False, "axes.spines.right": False})

def obs(kind, label):
    return np.load(OBS / f"{kind}_{label}.npz")

print("repository root:", ROOT)

## 1. XL-BOMD trajectories

Both runs: AM1, Niklasson XL-BOMD with k = 6, NVE, 500 K initial temperature, 0.25 fs step.
ISO17: 4000 steps, all frames used, RMSD relative to frame 0.
MD17 aspirin: 10000 steps, first 2000 discarded as equilibration, RMSD relative to the mean geometry.

### Summary numbers (computed from the data)

In [ ]:
rows = []
for label, name in SYSTEMS.items():
    r, f, e, s = obs("rmsd", label), obs("rmsf", label), obs("energy", label), obs("pmf", label)
    sym, y = f["symbols"], f["XL-BOMD"]
    dE = e["XL-BOMD"] * EV_TO_KCAL
    win = max(5, len(dE) // 20)
    run = np.convolve(dE, np.ones(win) / win, mode="valid")
    rows.append([
        label, int(r["discard"]), str(r["reference"]),
        f"{r['XL-BOMD'][len(r['XL-BOMD'])//40:].min():.2f}-{r['XL-BOMD'].max():.2f}",
        f"{y[sym != 'H'].max():.2f}", f"{y[sym == 'H'].max():.2f}",
        f"{dE.min():+.3f} / {dE.max():+.3f}", f"{run.min():+.3f} / {run.max():+.3f}",
        f"{s['XL-BOMD'].max() * EV_TO_KCAL:.2f}",
    ])
head = ["system", "discarded", "RMSD ref", "RMSD (Å)", "RMSF C,O max (Å)", "RMSF H max (Å)",
        "ΔE_tot min/max (kcal/mol)", "running mean (kcal/mol)", "E-surface max (kcal/mol)"]
w = [max(len(h), *(len(r[i]) if isinstance(r[i], str) else len(str(r[i])) for r in rows)) for i, h in enumerate(head)]
print("  ".join(h.ljust(w[i]) for i, h in enumerate(head)))
for r in rows:
    print("  ".join(str(v).ljust(w[i]) for i, v in enumerate(r)))
print("\nRMSD range excludes the first 2.5% of frames (start-up transient).")

### RMSD

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, (label, name) in zip(axes, SYSTEMS.items()):
    d = obs("rmsd", label)
    ax.plot(d["axis"], d["XL-BOMD"], color="#16A085", lw=1.2)
    ax.set(title=f"{name}\nreference: {d['reference']}", xlabel="MD step", ylabel="RMSD (Å)")
    ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

### Per-atom RMSF (shaded by element, in trajectory atom order)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, (label, name) in zip(axes, SYSTEMS.items()):
    d = obs("rmsf", label)
    idx, y, sym = d["axis"], d["XL-BOMD"], [str(s) for s in d["symbols"]]
    ax.plot(idx, y, "o-", color="#16A085", ms=3, lw=1.2)
    top = y.max() * 1.25
    start = 0
    for i in range(1, len(sym) + 1):
        if i == len(sym) or sym[i] != sym[start]:
            c = ELEMENT_COLORS.get(sym[start], "0.5")
            ax.axvspan(start + 0.5, i + 0.5, color=c, alpha=0.08)
            ax.text((start + i + 1) / 2, top * 0.95, sym[start], color=c, ha="center", va="top")
            start = i
    ax.set(title=name, xlabel="atom index", ylabel="RMSF (Å)", ylim=(0, top), xlim=(0.5, len(idx) + 0.5))
plt.tight_layout(); plt.show()

### NVE energy conservation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 5.5), sharex="col")
for col, (label, name) in enumerate(SYSTEMS.items()):
    d = obs("energy", label)
    t, dE = d["axis"], d["XL-BOMD"] * EV_TO_KCAL
    win = max(5, len(dE) // 20)
    run = np.convolve(dE, np.ones(win) / win, mode="valid")
    axes[0, col].plot(t, dE, color="#16A085", lw=0.6)
    axes[0, col].set(title=name, ylabel=r"$\Delta E_\mathrm{tot}$ (kcal/mol)")
    axes[1, col].plot(t[win // 2: win // 2 + len(run)], run, color="#16A085", lw=1.4)
    axes[1, col].set(xlabel="MD step", ylabel=f"running mean ({win} steps)")
    for ax in axes[:, col]:
        ax.axhline(0, color="0.5", lw=0.7, ls="--")
plt.tight_layout(); plt.show()

### Potential energy vs RMSD (5th percentile per RMSD bin, smoothed)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, name in SYSTEMS.items():
    d = obs("pmf", label)
    ax.plot(d["axis"], d["XL-BOMD"] * EV_TO_KCAL, lw=1.8, label=name)
ax.set(xlabel="RMSD (Å)", ylabel="energy (kcal/mol)")
ax.legend(); plt.tight_layout(); plt.show()

### Trajectory frame viewer

Change `label` and `frame` to look at any frame. Bonds are drawn between atoms closer than
1.2 × the sum of their covalent radii.

In [ ]:
def read_frame(path, frame):
    with open(path) as f:
        n = int(f.readline())
        for _ in range(frame * (n + 2)):
            f.readline()
        f.seek(0)
        lines = f.read().splitlines()
    block = lines[frame * (n + 2) + 2: frame * (n + 2) + 2 + n]
    sym = [l.split()[0] for l in block]
    xyz = np.array([[float(v) for v in l.split()[1:4]] for l in block])
    return sym, xyz

RADII = {"H": 0.31, "C": 0.76, "N": 0.71, "O": 0.66}
label, frame = "md17", 0
sym, X = read_frame(MD / f"{label}_run.0.xyz", frame)
X = X - X.mean(axis=0)
X = X @ np.linalg.svd(X, full_matrices=False)[2].T   # principal axes: molecule faces the viewer
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection="3d")
for i in range(len(sym)):
    for j in range(i + 1, len(sym)):
        if np.linalg.norm(X[i] - X[j]) < 1.2 * (RADII[sym[i]] + RADII[sym[j]]):
            ax.plot(*zip(X[i], X[j]), color="0.6", lw=2)
ax.scatter(*X.T, s=[200 if s != "H" else 60 for s in sym],
           c=[ELEMENT_COLORS.get(s, "0.5") for s in sym], depthshade=False, edgecolors="k")
ax.set_title(f"{label}, frame {frame}")
ax.view_init(elev=75, azim=-90)
ax.set_axis_off(); ax.set_box_aspect(np.maximum(np.ptp(X, axis=0), 0.5))
plt.tight_layout(); plt.show()

## 2. QLSA experiment

With the linear SCF surrogate $D(P)=(1-\gamma)D^\star+\gamma P$, the XL-BOMD electronic update over
$T$ steps is one linear system $Mx=b$. `scripts/qlsa/run_qlsa_experiment.py` solves it classically
(step by step) and with a matrix-level emulation of a QSVT quantum linear solver
($\varepsilon = 10^{-8}$), for two recurrences:

* **paper**: the two-step update of the manuscript, Niklasson et al. (2009) Eq. 24,
  $P_{n+1}=2P_n-P_{n-1}+(2-\alpha)(D_n-P_n)-\alpha(D_{n-1}-P_{n-1})$,
  without ($\alpha = 0$) and with ($\alpha = 0.5$) dissipation;
* **pyseqm**: the k = 6 scheme PySEQM integrates, under the same surrogate.

The inputs are AM1 densities from the first 200 frames of each trajectory. The emulation models
the solver's polynomial approximation exactly; it does not model gate noise, success probability,
or readout sampling.

In [ ]:
def load_qlsa(label):
    return json.load(open(QLSA / f"qlsa_{label}.json"))

def tag(r):
    return f"paper α={r['alpha']}" if r["scheme"] == "paper" else f"PySEQM k={r['k']}"

for label in SYSTEMS:
    q = load_qlsa(label)
    print(f"\n{label}: {q['meta']['source']}")
    print(f"{'scheme':16s} {'γ':>5s} {'T':>5s} {'κ_eff':>10s} {'queries':>10s} {'QLSA vs classical':>18s} {'surrogate vs SCF':>17s}")
    for r in q["rows"]:
        if r["gamma"] != 0.5:
            continue
        dev = f"{r['P_rel_frobenius']:.1e}" if r["feasible"] else "infeasible"
        full = r.get("surrogate_vs_scf_frames", 0) >= r["T"] - 8      # only 200 frames were exported
        sur = f"{100 * r['surrogate_vs_scf_rel']:.1f}%" if r["feasible"] and full else "-"
        print(f"{tag(r):16s} {r['gamma']:5.2f} {r['T']:5d} {r['kappa_eff']:10.3e} {r['degree']:10.3e} {dev:>18s} {sur:>17s}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
q = load_qlsa("iso17")
series = {}
for r in q["rows"]:
    if r["gamma"] == 0.5:
        series.setdefault(tag(r), []).append(r)
for name, rs in series.items():
    rs = sorted(rs, key=lambda r: r["T"])
    T = [r["T"] for r in rs]
    axes[0].loglog(T, [r["kappa_eff"] for r in rs], "o-", label=name)
    ok = [r for r in rs if r["feasible"]]
    axes[1].loglog([r["T"] for r in ok], [r["P_rel_frobenius"] for r in ok], "o-", label=name)
    sur = [r for r in ok if r.get("surrogate_vs_scf_frames", 0) >= r["T"] - 8]
    axes[2].plot([r["T"] for r in sur], [100 * r["surrogate_vs_scf_rel"] for r in sur], "o-", label=name)
axes[0].set(xlabel="T (steps)", ylabel="effective condition number", title="conditioning (γ = 0.5)")
axes[1].set(xlabel="T (steps)", ylabel="relative Frobenius deviation", title="QSVT vs classical solve")
axes[2].set(xlabel="T (steps)", ylabel="% relative deviation", title="surrogate vs AM1 SCF densities")
axes[0].legend(); plt.tight_layout(); plt.show()

Long windows (T up to 4000). The condition number depends only on the recurrence, not on the molecule, so this run uses synthetic input.

In [ ]:
q = json.load(open(QLSA / "qlsa_long_windows_synthetic.json"))
fig, ax = plt.subplots(figsize=(6, 3.8))
for name in sorted({tag(r) for r in q["rows"]}):
    rs = sorted([r for r in q["rows"] if tag(r) == name], key=lambda r: r["T"])
    ax.loglog([r["T"] for r in rs], [r["kappa_eff"] for r in rs], "o-", label=name)
ax.set(xlabel="T (steps)", ylabel="effective condition number", title="long windows, γ = 0.5")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Periodic re-linearization

`scripts/qlsa/run_windowed_experiment.py` splits 1000 frames (250 fs) into windows of W steps.
At each window start $D^\star$ is set to the SCF density of that frame, and the next W steps come
from one linear solve (PySEQM's k = 6 recurrence, γ = 0.5). The history is carried over from the
previous window, so errors at a window's end enter the next one.

In [ ]:
for label in SYSTEMS:
    w = json.load(open(QLSA / f"windowed_{label}.json"))
    print(f"\n{label}: {w['meta']['source']}")
    print(f"{'W':>4s} {'SCFs':>5s} {'mean dev %':>11s} {'max dev %':>10s} {'κ_eff':>7s} {'degree':>7s} {'QSVT vs classical':>18s}")
    for r in w["rows"]:
        print(f"{r['W']:4d} {r['n_scf']:5d} {100*r['rel_frobenius_all']:11.2f} {100*r['max_frame_error']:10.2f} "
              f"{r['kappa_eff_window']:7.1f} {r['qsvt_degree_window']:7d} {r['qsvt_vs_classical_max']:18.1e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
colors = ["#1b6ca8", "#e08a1e", "#3a9b5c", "#b03a48"]
for ax, label in zip(axes, SYSTEMS):
    rows = json.load(open(QLSA / f"windowed_{label}.json"))["rows"]
    for c, W in reversed(list(zip(colors, [5, 10, 20, 100]))):
        r = next(r for r in rows if r["W"] == W)
        e = 100 * np.asarray(r["error_per_frame"])
        ax.plot(0.25 * np.arange(len(e)), e, color=c, lw=0.6, label=f"W = {W}")
    ax.set(title=SYSTEMS[label], xlabel="time (fs)", ylabel="deviation from SCF density (%)")
    h, l = ax.get_legend_handles_labels(); ax.legend(h[::-1], l[::-1], ncol=4, fontsize=8)
for label, mk in zip(SYSTEMS, "os"):
    rows = json.load(open(QLSA / f"windowed_{label}.json"))["rows"]
    axes[2].semilogx([r["W"] for r in rows], [100 * r["rel_frobenius_all"] for r in rows], mk + "-", label=label)
axes[2].set(xlabel="window length W (steps)", ylabel="overall deviation (%)")
axes[2].legend(); plt.tight_layout(); plt.show()

## 4. MAE evaluation (AM1 vs DFT labels)

In [ ]:
files = sorted(MAE.glob("*.json"))
if not files:
    print("No MAE results yet. Run ./download_data.sh and then ./run_all.sh mae.")
for f in files:
    res = json.load(open(f))
    print(f"\n{f.name}  (method {res['method']}, {res['offset_note']})")
    print(f"{'set':32s} {'energy MAE':>12s} {'force MAE':>14s} {'converged':>10s}")
    for r in res["rows"]:
        print(f"{r['name']:32s} {r['energy_mae_kcal']:9.3f} kcal/mol {r['force_mae_kcal_A']:8.3f} kcal/mol/Å "
              f"{r['n_converged']:>5d}/{r['n_test']}")

In [ ]:
# Published ML-potential MAEs (kcal/mol and kcal/mol/Å), as cited in the manuscript:
# PhysNet, SchNet from their papers (ISO17) or the PaiNN paper (MD17); TAIP variants from the TAIP paper.
BEST_ML = {  # best published value per set: (energy, force)
    "ISO17 known mol / unk conf": (0.10, 0.12), "ISO17 unknown mol / unk conf": (0.662, 0.684),
    "MD17 aspirin": (0.144, 0.330), "MD17 ethanol": (0.057, 0.160), "MD17 malonaldehyde": (0.091, 0.297),
    "MD17 naphthalene": (0.093, 0.072), "MD17 salicylic": (0.105, 0.193), "MD17 C6H5CH3": (0.097, 0.090),
    "MD17 uracil": (0.090, 0.130),
}
print(f"{'method':9s} {'set':32s} {'E':>7s} {'best ML E':>9s} {'ratio':>6s} {'F':>7s} {'best ML F':>9s} {'ratio':>6s}")
for f in sorted(MAE.glob("*.json")):
    res = json.load(open(f))
    for r in res["rows"]:
        if r["name"] in BEST_ML:
            e, fm = BEST_ML[r["name"]]
            print(f"{res['method']:9s} {r['name']:32s} {r['energy_mae_kcal']:7.2f} {e:9.3f} {r['energy_mae_kcal']/e:6.0f} "
                  f"{r['force_mae_kcal_A']:7.2f} {fm:9.3f} {r['force_mae_kcal_A']/fm:6.0f}")

## 5. GFN2-xTB runs (if present)

`./run_all.sh xtb` repeats the trajectory, QLSA, re-linearization and MAE steps with GFN2-xTB
(tblite) instead of AM1. These cells show the results once the files exist.

In [ ]:
for label in SYSTEMS:
    f = MD / f"{label}_gfn2_energies.npz"
    if not f.exists():
        print(f"{label}: no GFN2-xTB trajectory yet (./run_all.sh xtb)"); continue
    e = np.load(f)
    dE = (e["E_tot_eV"] - e["E_tot_eV"][0]) * EV_TO_KCAL
    print(f"{label}: {len(dE)-1} steps, ΔE_tot {dE.min():+.3f} / {dE.max():+.3f} kcal/mol, T {e['T_K'].min():.0f}-{e['T_K'].max():.0f} K")
    plt.plot(e["t_fs"], dE, lw=0.7, label=label)
if plt.gca().lines:
    plt.xlabel("time (fs)"); plt.ylabel("ΔE_tot (kcal/mol)"); plt.title("GFN2-xTB NVE"); plt.legend(); plt.show()
for label in SYSTEMS:
    for kind in ("qlsa", "windowed"):
        f = QLSA / f"{kind}_{label}_gfn2.json"
        if not f.exists():
            continue
        rows = json.load(open(f))["rows"]
        print(f"\n{f.name}")
        if kind == "qlsa":
            for r in rows:
                if r["gamma"] == 0.5:
                    s = r.get("surrogate_vs_scf_rel")
                    print(f"  {tag(r):14s} T={r['T']:4d} κ_eff={r['kappa_eff']:8.1f} dev={r['P_rel_frobenius']:.1e} "
                          f"drift={'-' if s is None or r['surrogate_vs_scf_frames'] < r['T'] - 8 else f'{100*s:.1f}%'}")
        else:
            for r in rows:
                print(f"  W={r['W']:4d} mean dev {100*r['rel_frobenius_all']:5.2f}%  max {100*r['max_frame_error']:5.2f}%")